# Global Earthquake Database 2000-2025
### 46,000+ Seismic Events | Complete World Analysis + Machine Learning
**Author:** Hassan Ali | [Kaggle: hassanali789](https://www.kaggle.com/hassanali789)

This notebook provides a complete analysis of global earthquake data (Magnitude 5.0+) from 2000 to 2025.
We explore temporal trends, geographic hotspots, depth analysis, magnitude distributions, and build a magnitude class prediction model.

**Source:** USGS Earthquake Hazards Program

---


## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

print("All libraries loaded successfully!")

## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/global-earthquake-database-2000-2025/global_natural_disasters_2000_2025.csv",
                 parse_dates=["date"])

df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month

print(f"Shape          : {df.shape}")
print(f"Date range     : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Total events   : {len(df):,}")
print(f"Magnitude range: {df['magnitude'].min()} to {df['magnitude'].max()}")
print(f"Depth range    : {df['depth_km'].min():.1f} to {df['depth_km'].max():.1f} km")
print(f"\nMissing values:")
print(df.isnull().sum())
df.head(10)

## 3. Statistical Summary

In [ ]:
print("=== Descriptive Statistics ===")
print(df[["magnitude","depth_km","latitude","longitude"]].describe().round(2))

print("\n=== Magnitude Class Breakdown ===")
bins   = [5, 6, 7, 10]
labels = ["M5-6 (Light)", "M6-7 (Strong)", "M7+ (Major/Great)"]
df["mag_class"] = pd.cut(df["magnitude"], bins=bins, labels=labels)
print(df["mag_class"].value_counts().sort_index())

print("\n=== Depth Category Breakdown ===")
df["depth_cat"] = pd.cut(df["depth_km"],
                          bins=[-1, 70, 300, 1000],
                          labels=["Shallow (<70km)", "Intermediate (70-300km)", "Deep (>300km)"])
print(df["depth_cat"].value_counts())

## 4. Annual Earthquake Frequency (2000-2025)

In [ ]:
yearly = df.groupby("year").size().reset_index(name="count")

fig, ax = plt.subplots()
ax.bar(yearly["year"], yearly["count"],
       color="#E53935", edgecolor="white", linewidth=0.3)
ax.plot(yearly["year"], yearly["count"].rolling(3).mean(),
        color="#B71C1C", linewidth=2, linestyle="--", label="3-year rolling average")
ax.set_title("Annual Earthquake Frequency M5.0+ (2000-2025)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Earthquakes")
ax.set_xticks(range(2000, 2026, 2))
ax.legend()
plt.tight_layout()
plt.savefig("earthquake_frequency.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Most active year : {yearly.loc[yearly['count'].idxmax(), 'year']} ({yearly['count'].max():,} events)")
print(f"Least active year: {yearly.loc[yearly['count'].idxmin(), 'year']} ({yearly['count'].min():,} events)")

## 5. Global Earthquake Map

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
scatter = ax.scatter(
    df["longitude"], df["latitude"],
    s=df["magnitude"] ** 2 * 0.3,
    c=df["magnitude"], cmap="YlOrRd",
    alpha=0.25, linewidths=0
)
cbar = plt.colorbar(scatter, ax=ax, label="Magnitude", shrink=0.7)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.axhline(0, color="white", linewidth=0.4, linestyle="--", alpha=0.4)
ax.axvline(0, color="white", linewidth=0.4, linestyle="--", alpha=0.4)
ax.set_facecolor("#0D1B2A")
fig.patch.set_facecolor("#0D1B2A")
ax.tick_params(colors="white")
cbar.ax.yaxis.label.set_color("white")
cbar.ax.tick_params(colors="white")
ax.set_title("Global Earthquake Map — M5.0+ (2000-2025)", fontsize=14,
             fontweight="bold", pad=12, color="white")
ax.set_xlabel("Longitude", color="white")
ax.set_ylabel("Latitude", color="white")
plt.tight_layout()
plt.savefig("earthquake_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Magnitude Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["magnitude"], bins=40, color="#E53935", edgecolor="white", linewidth=0.3)
axes[0].axvline(df["magnitude"].mean(), color="black", linestyle="--",
                label=f"Mean: {df['magnitude'].mean():.2f}")
axes[0].set_title("Magnitude Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Count")
axes[0].legend()

class_counts = df["mag_class"].value_counts().sort_index()
colors = ["#FFCDD2", "#EF9A9A", "#E53935"]
bars = axes[1].bar(class_counts.index, class_counts.values,
                   color=colors, edgecolor="white", linewidth=0.3)
axes[1].bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9)
axes[1].set_title("Events by Magnitude Class", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("magnitude_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Depth Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["depth_km"].clip(0, 500), bins=50,
             color="#1565C0", edgecolor="white", linewidth=0.3)
axes[0].axvline(70,  color="orange", linestyle="--", linewidth=1.5, label="Shallow limit (70km)")
axes[0].axvline(300, color="red",    linestyle="--", linewidth=1.5, label="Intermediate limit (300km)")
axes[0].set_title("Earthquake Depth Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Depth (km)")
axes[0].set_ylabel("Count")
axes[0].legend()

depth_counts = df["depth_cat"].value_counts()
axes[1].bar(depth_counts.index, depth_counts.values,
            color=["#EF5350", "#FFA726", "#42A5F5"],
            edgecolor="white", linewidth=0.3)
axes[1].set_title("Events by Depth Category", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=10)

plt.tight_layout()
plt.savefig("depth_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("Shallow earthquakes (<70km) are most destructive.")
print(f"Shallow  : {(df['depth_km'] < 70).sum():,} events ({(df['depth_km'] < 70).mean()*100:.1f}%)")
print(f"Deep     : {(df['depth_km'] > 300).sum():,} events ({(df['depth_km'] > 300).mean()*100:.1f}%)")

## 8. Depth vs Magnitude Relationship

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df["depth_km"], df["magnitude"], alpha=0.08, s=4, c="#E53935")
ax.set_xlim(0, 700)
ax.set_ylim(5, 10)
ax.set_title("Earthquake Depth vs Magnitude", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Depth (km)")
ax.set_ylabel("Magnitude")
ax.axvline(70,  color="orange", linestyle="--", linewidth=1, label="Shallow limit (70km)")
ax.axvline(300, color="red",    linestyle="--", linewidth=1, label="Intermediate limit (300km)")
ax.legend()

corr = df[["depth_km", "magnitude"]].corr().iloc[0, 1]
ax.text(0.98, 0.05, f"Correlation: {corr:.3f}",
        transform=ax.transAxes, ha="right", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.savefig("depth_vs_magnitude.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Monthly and Seasonal Patterns

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

monthly = df.groupby("month").size()
axes[0].bar(month_names, monthly.values, color="#7B1FA2", edgecolor="white", linewidth=0.3)
axes[0].set_title("Monthly Earthquake Frequency", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Total Events (2000-2025)")
axes[0].tick_params(axis="x", rotation=30)

monthly_mag = df.groupby("month")["magnitude"].mean()
axes[1].bar(month_names, monthly_mag.values, color="#AB47BC", edgecolor="white", linewidth=0.3)
axes[1].set_ylim(5.2, 5.5)
axes[1].set_title("Average Magnitude by Month", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Mean Magnitude")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("seasonal_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Magnitude Heatmap — Latitude vs Month

In [ ]:
df["lat_bin"] = pd.cut(df["latitude"], bins=range(-90, 91, 15)).astype(str)

pivot = df.pivot_table(values="magnitude", index="lat_bin", columns="month", aggfunc="mean")
pivot.columns = month_names

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Mean Magnitude"}, ax=ax)
ax.set_title("Mean Magnitude by Latitude Band and Month", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Month")
ax.set_ylabel("Latitude Band")
plt.tight_layout()
plt.savefig("magnitude_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Top 15 Strongest Earthquakes (2000-2025)

In [ ]:
top15 = (
    df.nlargest(15, "magnitude")
    [["date", "location", "latitude", "longitude", "magnitude", "depth_km"]]
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 6))

def make_label(r):
    name = str(r["location"] or "Unknown Location")[:35]
    return "{} ({})".format(name, r["date"].year)

labels = top15.apply(make_label, axis=1)
colors = plt.cm.YlOrRd(np.linspace(0.4, 1.0, 15))[::-1]

bars = ax.barh(labels, top15["magnitude"],
               color=colors, edgecolor="white", linewidth=0.3)
ax.bar_label(bars, fmt="M%.1f", padding=3, fontsize=9)
ax.set_title("Top 15 Strongest Earthquakes (2000-2025)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Magnitude")
ax.invert_yaxis()
ax.set_xlim(8.0, 9.5)
plt.tight_layout()
plt.savefig("top15_strongest.png", dpi=150, bbox_inches="tight")
plt.show()

print(top15.to_string(index=False))

## 12. Feature Engineering for Machine Learning

In [ ]:
df_ml = df.dropna(subset=["magnitude", "depth_km", "latitude", "longitude"]).copy()

df_ml["target_label"] = pd.cut(
    df_ml["magnitude"],
    bins=[5, 6, 7, 10],
    labels=["M5-6", "M6-7", "M7+"]
)
df_ml = df_ml.dropna(subset=["target_label"])

df_ml["lat_abs"]    = df_ml["latitude"].abs()
df_ml["lon_abs"]    = df_ml["longitude"].abs()
df_ml["is_pacific"] = ((df_ml["longitude"] > 100) | (df_ml["longitude"] < -60)).astype(int)
df_ml["is_shallow"] = (df_ml["depth_km"] < 70).astype(int)
df_ml["depth_log"]  = np.log1p(df_ml["depth_km"])
df_ml["lat_zone"]   = pd.cut(df_ml["latitude"],
                              bins=[-90, -30, 0, 30, 90],
                              labels=[0, 1, 2, 3]).astype(int)

le = LabelEncoder()
df_ml["target"] = le.fit_transform(df_ml["target_label"])

print(f"ML-ready rows : {len(df_ml):,}")
print(f"Class balance :")
print(df_ml["target_label"].value_counts().sort_index())
df_ml[["latitude", "longitude", "depth_km", "magnitude", "target_label"]].head()

## 13. Model Training — Predicting Magnitude Class

In [ ]:
features = ["latitude", "longitude", "depth_km", "depth_log",
            "lat_abs", "lon_abs", "is_pacific", "is_shallow",
            "month", "year", "lat_zone"]

X = df_ml[features]
y = df_ml["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]:,}")
print(f"Test size : {X_test.shape[0]:,}")
print()

models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []
trained = {}

for name, model in models.items():
    Xtr = X_train_s if name == "Logistic Regression" else X_train
    Xte = X_test_s  if name == "Logistic Regression" else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    acc = accuracy_score(y_test, preds)
    results.append({"Model": name, "Accuracy": round(acc, 4)})
    trained[name] = (model, preds)
    print(f"{name:25s} -> Accuracy: {acc:.4f}")

## 14. Model Comparison and Feature Importance

In [ ]:
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
best_name  = results_df.iloc[0]["Model"]
best_preds = trained[best_name][1]
best_model = trained[best_name][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(results_df["Model"], results_df["Accuracy"],
                   color=["#43A047", "#1E88E5", "#FB8C00"],
                   edgecolor="white", linewidth=0.3)
axes[0].bar_label(bars, fmt="%.4f", padding=3, fontsize=10)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title("Model Accuracy Comparison", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Accuracy")
axes[0].tick_params(axis="x", rotation=10)

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=features).sort_values()
    axes[1].barh(importances.index, importances.values, color="#66BB6A")
    axes[1].set_title(f"Feature Importance — {best_name}", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Importance Score")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best Model: {best_name}")
print(f"Accuracy  : {results_df.iloc[0]['Accuracy']}")

## 15. Confusion Matrix and Classification Report

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, best_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_,
            yticklabels=le.classes_, ax=ax)
ax.set_title(f"Confusion Matrix — {best_name}", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("Classification Report:")
print(classification_report(y_test, best_preds, target_names=le.classes_))

## 16. Key Findings and Conclusions

### Seismic Patterns
- The Pacific Ring of Fire is clearly visible on the global map, accounting for roughly 80% of all M5.0+ earthquakes
- Shallow earthquakes (less than 70km depth) make up the majority of events and are the most destructive
- Deep earthquakes (greater than 300km) rarely cause significant surface damage despite high magnitudes

### Temporal Trends
- Earthquake frequency has remained relatively stable over 25 years
- No significant seasonal pattern — earthquakes are uniformly distributed across months
- The 2011 Tohoku earthquake (M9.1) is the strongest event in this dataset

### Machine Learning
- Geographic location (latitude, longitude) and depth are the strongest magnitude predictors
- Random Forest and Gradient Boosting outperform Logistic Regression significantly
- Class imbalance (very few M7+ events) is the main challenge for the classifier

---

Dataset by Hassan Ali | hassanali789 on Kaggle
Source: USGS Earthquake Hazards Program
